# LeetCode 1465 — Maximum Area of a Piece of Cake After Horizontal and Vertical Cuts bằng Python

Link đề: https://leetcode.com/problems/maximum-area-of-a-piece-of-cake-after-horizontal-and-vertical-cuts/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def maxArea(self, h: int, w: int, horizontalCuts: list[int], verticalCuts: list[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Thấy được mỗi miếng bánh là **tích** của một khoảng theo chiều cao và một khoảng theo chiều rộng, nên hai chiều **độc lập** với nhau.
2. Suy ra đáp án = (khoảng cách lớn nhất giữa hai nhát cắt ngang liền kề) × (khoảng cách lớn nhất giữa hai nhát cắt dọc liền kề).
3. Thành thạo kỹ thuật **sắp xếp + duyệt cặp liền kề**, nhớ thêm **biên `0` và `h`/`w`**.
4. Biết cách tìm **khoảng trống lớn nhất trong O(n)** bằng chia xô (bucket / chuồng bồ câu) — mở rộng của bài 164 Maximum Gap.
5. Lấy **modulo đúng lúc**: so sánh trên giá trị thật, chỉ `% (10^9+7)` ở kết quả cuối.

## 2. Đề bài & ví dụ

Có một chiếc bánh hình chữ nhật cao `h`, rộng `w`. Cho hai mảng:
- `horizontalCuts[i]`: khoảng cách từ **mép trên** bánh tới nhát cắt ngang thứ `i`;
- `verticalCuts[j]`: khoảng cách từ **mép trái** bánh tới nhát cắt dọc thứ `j`.

Sau khi cắt tại tất cả các vị trí, trả về **diện tích lớn nhất** của một miếng bánh, lấy modulo `10^9 + 7`.

**Ví dụ 1:** `h = 5, w = 4, horizontalCuts = [1,2,4], verticalCuts = [1,3]` → `4`.

**Ví dụ 2:** `h = 5, w = 4, horizontalCuts = [3,1], verticalCuts = [1]` → `6`.

**Ví dụ 3:** `h = 5, w = 4, horizontalCuts = [3], verticalCuts = [3]` → `9`.

**Ràng buộc:**
- `2 <= h, w <= 10^9`
- `1 <= horizontalCuts.length <= min(h − 1, 10^5)`, `1 <= verticalCuts.length <= min(w − 1, 10^5)`
- `1 <= horizontalCuts[i] < h`, `1 <= verticalCuts[i] < w`
- Các phần tử trong mỗi mảng **phân biệt** (nhưng **chưa sắp xếp**).

In [1]:
import random
import time
from typing import List

MOD = 10**9 + 7
examples = [
    (5, 4, [1, 2, 4], [1, 3], 4),
    (5, 4, [3, 1], [1], 6),
    (5, 4, [3], [3], 9),
]
for h, w, hc, vc, want in examples:
    print(f"h={h}, w={w}, horizontalCuts={hc}, verticalCuts={vc} -> kỳ vọng {want}")

h=5, w=4, horizontalCuts=[1, 2, 4], verticalCuts=[1, 3] -> kỳ vọng 4
h=5, w=4, horizontalCuts=[3, 1], verticalCuts=[1] -> kỳ vọng 6
h=5, w=4, horizontalCuts=[3], verticalCuts=[3] -> kỳ vọng 9


## 3. Kiến thức nền

### 3.1 `sorted()` và `list.sort()`

`sorted(a)` trả về **danh sách mới** đã sắp; `a.sort()` sắp **tại chỗ** và trả về `None`. Ở bài này các nhát cắt cho vào **không theo thứ tự**, nên phải sắp trước khi tính khoảng cách liền kề.

In [2]:
a = [3, 1, 4]
b = sorted(a)
print("a =", a, "| sorted(a) =", b)
r = a.sort()
print("sau a.sort(): a =", a, "| giá trị trả về =", r)

a = [3, 1, 4] | sorted(a) = [1, 3, 4]
sau a.sort(): a = [1, 3, 4] | giá trị trả về = None


### 3.2 Thêm biên rồi lấy hiệu các phần tử liền kề

Các "đường ranh giới" theo chiều cao gồm mép trên `0`, các nhát cắt, và mép dưới `h`. Ghép `[0] + sorted(cuts) + [h]` rồi lấy hiệu từng cặp kề nhau. `zip(p, p[1:])` sinh các cặp `(p[0],p[1]), (p[1],p[2]), …`.

In [3]:
h, cuts = 5, [3, 1]
p = [0] + sorted(cuts) + [h]
print("ranh giới:", p)
print("các cặp :", list(zip(p, p[1:])))
print("khoảng  :", [b - a for a, b in zip(p, p[1:])])

ranh giới: [0, 1, 3, 5]
các cặp : [(0, 1), (1, 3), (3, 5)]
khoảng  : [1, 2, 2]


### 3.3 `max()` trên generator và tham số `default`

`max(biểu_thức for …)` không cần tạo list trung gian. Nếu iterable có thể rỗng, dùng `max(…, default=0)` để tránh `ValueError`.

In [4]:
print(max(b - a for a, b in zip(p, p[1:])))
print(max([], default=0))

2
0


### 3.4 Modulo `10^9 + 7` và số nguyên lớn trong Python

Python có số nguyên **không tràn**, nên `(10^9 − 1) × (10^9 − 1)` vẫn chính xác. Đề yêu cầu trả về kết quả `% (10^9 + 7)`. Quan trọng: **so sánh / lấy max trên giá trị thật**, chỉ modulo ở bước cuối — vì modulo **không giữ thứ tự** (số lớn hơn có thể ra phần dư nhỏ hơn).

In [5]:
big = (10**9 - 1) * (10**9 - 1)
print("tích thật:", big, "| % MOD:", big % MOD)
x, y = 10**9 + 8, 10**9
print(f"{x} > {y} nhưng {x} % MOD = {x % MOD} < {y} % MOD = {y % MOD}")

tích thật: 999999998000000001 | % MOD: 64
1000000008 > 1000000000 nhưng 1000000008 % MOD = 1 < 1000000000 % MOD = 1000000000


### 3.5 Chia xô (bucket) — ý tưởng chuồng bồ câu

Có `m` điểm trên đoạn `[lo, hi]` ⇒ `m − 1` khoảng, tổng độ dài `hi − lo`. Khoảng lớn nhất **ít nhất bằng trung bình** `(hi − lo) / (m − 1)`. Nếu chia đoạn thành các xô có độ dài nhỏ hơn hoặc bằng trung bình đó, thì khoảng lớn nhất **không thể nằm trọn trong một xô** — chỉ cần so `min` của xô sau với `max` của xô trước. (Dùng ở Cách 3.)

In [6]:
pts = [0, 1, 2, 4, 5]
gaps = [b - a for a, b in zip(pts, pts[1:])]
print("khoảng:", gaps, "| trung bình =", (pts[-1] - pts[0]) / (len(pts) - 1), "| lớn nhất =", max(gaps))

khoảng: [1, 1, 2, 1] | trung bình = 1.25 | lớn nhất = 2


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1: `h = 5`, nhát ngang `[1, 2, 4]` ⇒ ranh giới `0,1,2,4,5` ⇒ các **dải ngang** cao `1, 1, 2, 1`. `w = 4`, nhát dọc `[1, 3]` ⇒ ranh giới `0,1,3,4` ⇒ các **dải dọc** rộng `1, 2, 1`.

Mỗi miếng = một dải ngang ∩ một dải dọc, diện tích = cao × rộng:

| cao \ rộng | 1 | 2 | 1 |
|---|---|---|---|
| **1** | 1 | 2 | 1 |
| **1** | 1 | 2 | 1 |
| **2** | 2 | **4** | 2 |
| **1** | 1 | 2 | 1 |

Ô lớn nhất là `4` = (cao lớn nhất `2`) × (rộng lớn nhất `2`). Bảng là "bảng cửu chương" của hai danh sách số dương ⇒ ô lớn nhất luôn là tích hai số lớn nhất.

In [7]:
def strips(L, cuts):
    p = [0] + sorted(cuts) + [L]
    return [b - a for a, b in zip(p, p[1:])]

h, w, hc, vc, _ = examples[0]
H, V = strips(h, hc), strips(w, vc)
print("cao các dải:", H, "| rộng các dải:", V)
for x in H:
    print([x * y for y in V])

cao các dải: [1, 1, 2, 1] | rộng các dải: [1, 2, 1]
[1, 2, 1]
[1, 2, 1]
[2, 4, 2]
[1, 2, 1]


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi miếng

**Ý tưởng:** tính tất cả chiều cao dải `H` và chiều rộng dải `V`, rồi thử **mọi cặp** `(x, y)`, lấy `max(x · y)`. Đây chính là bảng ở mục 4.

In [8]:
def max_area_brute(h, w, horizontalCuts, verticalCuts):
    H = strips(h, horizontalCuts)
    V = strips(w, verticalCuts)
    best = 0
    for x in H:
        for y in V:
            best = max(best, x * y)      # so sánh trên giá trị thật
    return best % MOD

for h, w, hc, vc, want in examples:
    got = max_area_brute(h, w, hc, vc)
    print(got, want, "✔" if got == want else "✘")

4 4 ✔
6 6 ✔
9 9 ✔


**Độ phức tạp:** sắp xếp `O(A log A + B log B)` (A, B là số nhát ngang, dọc), duyệt mọi miếng `O(A · B)`. Với `A = B = 10^5` có `10^10` miếng — không thể chạy.

**Nút thắt:** ta thử cả `(A+1)(B+1)` miếng, trong khi chỉ cần **một số lớn nhất mỗi chiều**. Cell dưới cho thấy thời gian tăng theo bình phương.

In [9]:
rng = random.Random(1465)
for n in (300, 1000, 2000):
    hc = rng.sample(range(1, 10**9), n)
    vc = rng.sample(range(1, 10**9), n)
    t = time.perf_counter()
    max_area_brute(10**9, 10**9, hc, vc)
    print(f"{n} nhát mỗi chiều: {(n + 1) ** 2:>9} miếng, {time.perf_counter() - t:.3f}s")

300 nhát mỗi chiều:     90601 miếng, 0.008s
1000 nhát mỗi chiều:   1002001 miếng, 0.091s


2000 nhát mỗi chiều:   4004001 miếng, 0.242s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** diện tích miếng `= x · y` với `x ∈ H`, `y ∈ V`, mọi số đều dương. Với số dương, **tăng một thừa số thì tích tăng**, nên:

> `max_{x∈H, y∈V} (x · y) = max(H) · max(V)`.

**Quy tắc:**
- **Nếu** muốn miếng lớn nhất **thì** chọn dải ngang cao nhất và dải dọc rộng nhất — chúng luôn giao nhau thành một miếng.
- **Nếu** cần khoảng lớn nhất giữa các điểm cắt **thì** sắp xếp, **thêm biên** `0` và `L`, lấy max hiệu liền kề.
- **Nếu** đề có modulo **thì** chỉ áp dụng sau phép nhân cuối cùng.

Hai chiều độc lập ⇒ bài 2D tách thành **hai bài 1D "khoảng trống lớn nhất"**.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + khoảng lớn nhất mỗi chiều

**Từng đoạn:**
1. Viết hàm `max_gap(L, cuts)`: sắp `cuts`, đi từ `prev = 0`, với mỗi nhát `c` cập nhật `best = max(best, c − prev)`, `prev = c`; cuối cùng so thêm `L − prev`.
2. Trả về `max_gap(h, horizontalCuts) * max_gap(w, verticalCuts) % MOD`.

In [10]:
def max_gap(L, cuts):
    best, prev = 0, 0
    for c in sorted(cuts):
        best = max(best, c - prev)
        prev = c
    return max(best, L - prev)          # dải cuối cùng tới mép L

def max_area_sort(h, w, horizontalCuts, verticalCuts):
    return max_gap(h, horizontalCuts) * max_gap(w, verticalCuts) % MOD

for h, w, hc, vc, want in examples:
    print(max_gap(h, hc), max_gap(w, vc), max_area_sort(h, w, hc, vc), want)

2 2 4 4
2 3 6 6
3 3 9 9


Code nộp LeetCode (Cách 2), viết gọn bằng `zip`:

In [11]:
class Solution:
    def maxArea(self, h: int, w: int, horizontalCuts: List[int], verticalCuts: List[int]) -> int:
        def gap(L, cuts):
            p = [0] + sorted(cuts) + [L]
            return max(b - a for a, b in zip(p, p[1:]))
        return gap(h, horizontalCuts) * gap(w, verticalCuts) % (10**9 + 7)

print([Solution().maxArea(h, w, hc, vc) for h, w, hc, vc, _ in examples])

[4, 6, 9]


**Độ phức tạp:** `O(A log A + B log B)` thời gian (sắp xếp chiếm chủ yếu), `O(A + B)` bộ nhớ cho bản sao đã sắp (dùng `.sort()` tại chỗ thì `O(1)` phụ trợ ngoài bộ nhớ của sort).

### 7.2 Cách 3 — Khoảng trống lớn nhất trong O(n) bằng chia xô

Không cần sắp toàn bộ. Với `m` điểm (gồm 2 biên) trên `[0, L]`:
1. `size = max(1, L // (m − 1))` — độ dài xô không vượt quá khoảng trung bình.
2. Mỗi điểm `x` vào xô `x // size`; mỗi xô chỉ nhớ `min` và `max`.
3. Duyệt các xô theo thứ tự, bỏ qua xô rỗng: khoảng ứng viên = `min(xô hiện tại) − max(xô không rỗng trước đó)`.

Vì khoảng lớn nhất `≥ L/(m−1) ≥ size`, còn hai điểm trong cùng một xô cách nhau `≤ size − 1`, nên khoảng lớn nhất luôn **vắt qua ranh giới xô** ⇒ chỉ cần so các xô liền kề không rỗng.

In [12]:
def max_gap_bucket(L, cuts):
    pts_count = len(cuts) + 2
    size = max(1, L // (pts_count - 1))
    nb = L // size + 1
    lo = [None] * nb
    hi = [None] * nb
    for x in [0, L, *cuts]:
        b = x // size
        if lo[b] is None:
            lo[b] = hi[b] = x
        else:
            if x < lo[b]: lo[b] = x
            if x > hi[b]: hi[b] = x
    best, prev_hi = 0, None
    for b in range(nb):
        if lo[b] is None:
            continue
        if prev_hi is not None:
            best = max(best, lo[b] - prev_hi)
        prev_hi = hi[b]
    return best

def max_area_bucket(h, w, horizontalCuts, verticalCuts):
    return max_gap_bucket(h, horizontalCuts) * max_gap_bucket(w, verticalCuts) % MOD

for h, w, hc, vc, want in examples:
    print(max_gap_bucket(h, hc), max_gap_bucket(w, vc), max_area_bucket(h, w, hc, vc), want)

2 2 4 4
2 3 6 6
3 3 9 9


Lưu ý số xô `nb = L // size + 1` xấp xỉ `m`, nên bộ nhớ `O(m)` chứ không phải `O(L)` dù `L` tới `10^9`. Code nộp LeetCode (Cách 3):

In [13]:
class Solution:
    def maxArea(self, h: int, w: int, horizontalCuts: List[int], verticalCuts: List[int]) -> int:
        def gap(L, cuts):
            size = max(1, L // (len(cuts) + 1))
            nb = L // size + 1
            lo, hi = [None] * nb, [None] * nb
            for x in [0, L, *cuts]:
                b = x // size
                if lo[b] is None:
                    lo[b] = hi[b] = x
                else:
                    lo[b] = min(lo[b], x)
                    hi[b] = max(hi[b], x)
            best, prev = 0, None
            for b in range(nb):
                if lo[b] is not None:
                    if prev is not None:
                        best = max(best, lo[b] - prev)
                    prev = hi[b]
            return best
        return gap(h, horizontalCuts) * gap(w, verticalCuts) % (10**9 + 7)

print([Solution().maxArea(h, w, hc, vc) for h, w, hc, vc, _ in examples])

[4, 6, 9]


**Độ phức tạp:** `O(A + B)` thời gian và bộ nhớ. Trên lý thuyết tốt hơn Cách 2, nhưng trong Python `sorted` viết bằng C nên Cách 2 thường **nhanh hơn trên thực tế**. Cách 3 có giá trị học thuật và hữu ích khi gặp bài 164 Maximum Gap (yêu cầu tuyến tính).

In [14]:
rng = random.Random(5)
hc = rng.sample(range(1, 10**9), 10**5)
vc = rng.sample(range(1, 10**9), 10**5)
for name, f in [("sort + max gap", max_area_sort), ("bucket O(n)", max_area_bucket)]:
    t = time.perf_counter()
    r = f(10**9, 10**9, hc, vc)
    print(f"{name:15s}: {r}  {time.perf_counter() - t:.3f}s")

sort + max gap : 73136436  0.072s


bucket O(n)    : 73136436  0.065s


## 8. TRACE từng bước

Trace `max_gap` cho chiều cao của ví dụ 2 (`h = 5`, `cuts = [3, 1]`) và chiều rộng (`w = 4`, `cuts = [1]`):

| bước | c | c − prev | best | prev |
|---|---|---|---|---|
| sắp xếp | `[1, 3]` | | 0 | 0 |
| 1 | 1 | 1 | 1 | 1 |
| 2 | 3 | 2 | 2 | 3 |
| biên | h = 5 | 2 | 2 | |

Chiều rộng: `[1]` ⇒ khoảng `1`, rồi biên `4 − 1 = 3` ⇒ `3`. Đáp án `2 × 3 = 6`.

In [15]:
def trace_max_gap(L, cuts, name):
    print(f"[{name}] L={L}, cuts={cuts} -> sắp xếp: {sorted(cuts)}")
    best, prev = 0, 0
    for i, c in enumerate(sorted(cuts), 1):
        d = c - prev
        best = max(best, d)
        print(f"  bước {i}: c={c}, c-prev={d}, best={best}, prev={c}")
        prev = c
    d = L - prev
    best = max(best, d)
    print(f"  biên  : L-prev={d}, best={best}")
    return best

def trace_max_area(h, w, hc, vc):
    a = trace_max_gap(h, hc, "ngang")
    b = trace_max_gap(w, vc, "dọc")
    print(f"=> {a} * {b} = {a * b} -> % MOD = {a * b % MOD}")

trace_max_area(5, 4, [3, 1], [1])

[ngang] L=5, cuts=[3, 1] -> sắp xếp: [1, 3]
  bước 1: c=1, c-prev=1, best=1, prev=1
  bước 2: c=3, c-prev=2, best=2, prev=3
  biên  : L-prev=2, best=2
[dọc] L=4, cuts=[1] -> sắp xếp: [1]
  bước 1: c=1, c-prev=1, best=1, prev=1
  biên  : L-prev=3, best=3
=> 2 * 3 = 6 -> % MOD = 6


In [16]:
trace_max_area(5, 4, [1, 2, 4], [1, 3])
print()
trace_max_area(10**9, 10**9, [2], [2])

[ngang] L=5, cuts=[1, 2, 4] -> sắp xếp: [1, 2, 4]
  bước 1: c=1, c-prev=1, best=1, prev=1
  bước 2: c=2, c-prev=1, best=1, prev=2
  bước 3: c=4, c-prev=2, best=2, prev=4
  biên  : L-prev=1, best=2
[dọc] L=4, cuts=[1, 3] -> sắp xếp: [1, 3]
  bước 1: c=1, c-prev=1, best=1, prev=1
  bước 2: c=3, c-prev=2, best=2, prev=3
  biên  : L-prev=1, best=2
=> 2 * 2 = 4 -> % MOD = 4

[ngang] L=1000000000, cuts=[2] -> sắp xếp: [2]
  bước 1: c=2, c-prev=2, best=2, prev=2
  biên  : L-prev=999999998, best=999999998
[dọc] L=1000000000, cuts=[2] -> sắp xếp: [2]
  bước 1: c=2, c-prev=2, best=2, prev=2
  biên  : L-prev=999999998, best=999999998
=> 999999998 * 999999998 = 999999996000000004 -> % MOD = 81


Trace Cách 3 (chia xô) trên `L = 20`, `cuts = [3, 17, 4, 9, 10]`:

In [17]:
def trace_bucket(L, cuts):
    m = len(cuts) + 2
    size = max(1, L // (m - 1))
    nb = L // size + 1
    buckets = [[] for _ in range(nb)]
    for x in [0, L, *cuts]:
        buckets[x // size].append(x)
    print(f"L={L}, m={m} điểm, size={size}, số xô={nb}")
    best, prev = 0, None
    for b, items in enumerate(buckets):
        rng_txt = f"[{b * size}, {(b + 1) * size - 1}]"
        if not items:
            print(f"  xô {b} {rng_txt}: rỗng")
            continue
        lo, hi = min(items), max(items)
        cand = None if prev is None else lo - prev
        if cand is not None:
            best = max(best, cand)
        print(f"  xô {b} {rng_txt}: {sorted(items)} min={lo} max={hi} ứng viên={cand} best={best}")
        prev = hi
    print("khoảng lớn nhất =", best, "| kiểm tra bằng sort:", max_gap(L, cuts))

trace_bucket(20, [3, 17, 4, 9, 10])

L=20, m=7 điểm, size=3, số xô=7
  xô 0 [0, 2]: [0] min=0 max=0 ứng viên=None best=0
  xô 1 [3, 5]: [3, 4] min=3 max=4 ứng viên=3 best=3
  xô 2 [6, 8]: rỗng
  xô 3 [9, 11]: [9, 10] min=9 max=10 ứng viên=5 best=5
  xô 4 [12, 14]: rỗng
  xô 5 [15, 17]: [17] min=17 max=17 ứng viên=7 best=7
  xô 6 [18, 20]: [20] min=20 max=20 ứng viên=3 best=7
khoảng lớn nhất = 7 | kiểm tra bằng sort: 7


## 9. Code nộp LeetCode

Bản khuyên dùng (sắp xếp + khoảng lớn nhất), đúng chữ ký `maxArea(self, h, w, horizontalCuts, verticalCuts)`:

In [18]:
class Solution:
    def maxArea(self, h: int, w: int, horizontalCuts: List[int], verticalCuts: List[int]) -> int:
        def gap(L: int, cuts: List[int]) -> int:
            cuts = sorted(cuts)
            best = max(cuts[0], L - cuts[-1])           # hai dải sát biên
            for a, b in zip(cuts, cuts[1:]):
                best = max(best, b - a)
            return best
        return gap(h, horizontalCuts) * gap(w, verticalCuts) % (10**9 + 7)

for h, w, hc, vc, want in examples:
    assert Solution().maxArea(h, w, hc, vc) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Mỗi miếng = (dải ngang) × (dải dọc).** Các nhát cắt ngang chia chiều cao thành các dải liên tiếp giữa hai ranh giới kề nhau (kể cả mép `0` và `h`); tương tự chiều rộng. Nhát cắt ngang cắt **xuyên suốt** bánh, nên mọi dải ngang giao với mọi dải dọc tạo thành một miếng. Vậy tập diện tích = `{x · y : x ∈ H, y ∈ V}`.
2. **Tích lớn nhất.** Với mọi `x ∈ H, y ∈ V`: `x ≤ max H`, `y ≤ max V`, đều dương ⇒ `x·y ≤ maxH · maxV`; và cặp `(maxH, maxV)` thực sự là một miếng ⇒ đạt được dấu bằng.
3. **Khoảng lớn nhất qua sắp xếp.** Sau khi sắp, hai ranh giới kề nhau trong danh sách chính là biên của một dải; không có ranh giới nào khác nằm giữa chúng.
4. **Chia xô.** Như mục 7.2: khoảng lớn nhất `≥ size` mà khoảng trong một xô `< size` ⇒ khoảng lớn nhất nằm giữa `max` một xô và `min` xô không rỗng kế tiếp.
5. **Modulo.** Lấy max trên giá trị thật rồi mới `% MOD` ⇒ đúng định nghĩa "diện tích lớn nhất, sau đó lấy modulo".

## 11. Phân tích Time / Space

Gọi `A = len(horizontalCuts)`, `B = len(verticalCuts)` (≤ `10^5`).

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê mọi miếng | `O(A log A + B log B + A·B)` | `O(A + B)` |
| 2. Sort + max gap | `O(A log A + B log B)` | `O(A + B)` (bản sao đã sắp) |
| 3. Chia xô | `O(A + B)` | `O(A + B)` |

`h, w` lớn tới `10^9` **không ảnh hưởng** độ phức tạp (Cách 3 dùng số xô ≈ số điểm, không phải `L`). Phép nhân cuối có kết quả tới `~10^18` — Python xử lý tự nhiên; trong C++/Java cần kiểu 64-bit.

## 12. Bộ test

Ví dụ đề, edge case (một nhát mỗi chiều, cắt sát mép, cắt mọi vị trí nguyên, `h = w = 10^9`, trường hợp cần modulo), đối chiếu ngẫu nhiên 3 cách + code nộp, test ràng buộc lớn.

In [19]:
ALL = [max_area_brute, max_area_sort, max_area_bucket,
       lambda h, w, a, b: Solution().maxArea(h, w, a, b)]

def check_all(h, w, hc, vc, want=None):
    res = [f(h, w, list(hc), list(vc)) for f in ALL]
    if want is None:
        want = res[0]
    assert all(r == want for r in res), (h, w, hc, vc, res)
    return want

for h, w, hc, vc, want in examples:
    check_all(h, w, hc, vc, want)

assert check_all(2, 2, [1], [1]) == 1                       # nhỏ nhất
assert check_all(10, 10, list(range(1, 10)), [5]) == 5       # cắt mọi vị trí theo chiều ngang
assert check_all(10, 7, [1], [6]) == 9 * 6                   # dải lớn nằm sát biên dưới / trái
assert check_all(10**9, 10**9, [1], [1]) == (10**9 - 1) ** 2 % MOD
assert check_all(10, 125000002, [8], [125000001]) == (8 * 125000001) % MOD   # = 1 sau modulo
print("Edge case: OK, ví dụ modulo ->", check_all(10, 125000002, [8], [125000001]))

Edge case: OK, ví dụ modulo -> 1


In [20]:
rng = random.Random(2023)
for _ in range(2000):
    h = rng.randint(2, 60)
    w = rng.randint(2, 60)
    hc = rng.sample(range(1, h), rng.randint(1, h - 1))
    vc = rng.sample(range(1, w), rng.randint(1, w - 1))
    check_all(h, w, hc, vc)
for _ in range(300):                                       # số lớn, ít nhát
    h = rng.randint(2, 10**9); w = rng.randint(2, 10**9)
    hc = list({rng.randint(1, h - 1) for _ in range(rng.randint(1, 30))})
    vc = list({rng.randint(1, w - 1) for _ in range(rng.randint(1, 30))})
    check_all(h, w, hc, vc)
print("Random 2300 test: OK")

Random 2300 test: OK


In [21]:
# Ràng buộc lớn: 10^5 nhát mỗi chiều
rng = random.Random(77)
hc = rng.sample(range(1, 10**9), 10**5)
vc = rng.sample(range(1, 10**9), 10**5)
t = time.perf_counter()
r = Solution().maxArea(10**9, 10**9, hc, vc)
print("kết quả:", r, f"| {time.perf_counter() - t:.3f}s")
assert r == max_area_bucket(10**9, 10**9, hc, vc)
# cắt mọi vị trí nguyên: h = 10^5 + 1
t = time.perf_counter()
print(Solution().maxArea(10**5 + 1, 10**5 + 1, list(range(1, 10**5 + 1)), list(range(1, 10**5 + 1))),
      f"{time.perf_counter() - t:.3f}s")

kết quả: 480954698 | 0.058s
1 0.020s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên hai biên `0` và `h`/`w`

Chỉ tính khoảng giữa các nhát cắt, bỏ quên dải sát mép.

In [22]:
def wrong_no_border(h, w, hc, vc):
    def gap(cuts):
        c = sorted(cuts)
        return max((b - a for a, b in zip(c, c[1:])), default=0)   # SAI: thiếu 0 và L
    return gap(hc) * gap(vc) % MOD

case = (10, 10, [1, 2], [5, 6])
print("SAI :", wrong_no_border(*case), "| đúng:", max_area_sort(*case))
print("Dải lớn nhất là 2..10 (cao 8) và 0..5 (rộng 5) — đều chạm biên.")

SAI : 1 | đúng: 40
Dải lớn nhất là 2..10 (cao 8) và 0..5 (rộng 5) — đều chạm biên.


**Sửa:** thêm `0` và `L` vào danh sách ranh giới (hoặc khởi tạo `best = max(cuts[0], L − cuts[-1])`).

### Lỗi 2 — Không sắp xếp các nhát cắt

Đề **không** đảm bảo mảng tăng dần (ví dụ 2 có `[3, 1]`). Lấy hiệu liền kề trên mảng chưa sắp sẽ ra khoảng âm hoặc khoảng "nhảy cóc".

In [23]:
def wrong_no_sort(h, w, hc, vc):
    def gap(L, cuts):
        p = [0] + cuts + [L]                                        # SAI: chưa sort
        return max(b - a for a, b in zip(p, p[1:]))
    return gap(h, hc) * gap(w, vc) % MOD

h, w, hc, vc, want = examples[1]
print("SAI :", wrong_no_sort(h, w, hc, vc), "| đúng:", want)
print("ranh giới chưa sắp [0, 3, 1, 5] -> hiệu 3, -2, 4: lấy nhầm 4.")

SAI : 12 | đúng: 6
ranh giới chưa sắp [0, 3, 1, 5] -> hiệu 3, -2, 4: lấy nhầm 4.


**Sửa:** `sorted(cuts)`.

### Lỗi 3 — Lấy modulo **trước** khi so sánh

Tính diện tích từng miếng rồi `% MOD` rồi mới lấy max. Modulo không giữ thứ tự nên có thể chọn sai miếng.

In [24]:
def wrong_mod_then_max(h, w, hc, vc):
    H, V = strips(h, hc), strips(w, vc)
    return max(x * y % MOD for x in H for y in V)                   # SAI: mod trước, max sau

case = (10, 125000002, [8], [125000001])
print("SAI :", wrong_mod_then_max(*case), "| đúng:", max_area_sort(*case))
print("Miếng lớn nhất 8 x 125000001 = 10^9 + 8 -> % MOD = 1; miếng 2 x 125000001 nhỏ hơn nhưng dư lớn hơn.")

SAI : 250000002 | đúng: 1
Miếng lớn nhất 8 x 125000001 = 10^9 + 8 -> % MOD = 1; miếng 2 x 125000001 nhỏ hơn nhưng dư lớn hơn.


**Sửa:** tìm max trên giá trị thật, chỉ `% MOD` ở cuối.

### Lỗi 4 — Quên modulo hoàn toàn

Python không tràn số nên code "chạy được" nhưng trả sai giá trị so với yêu cầu.

In [25]:
def wrong_no_mod(h, w, hc, vc):
    return max_gap(h, hc) * max_gap(w, vc)                          # SAI: thiếu % MOD

case = (10**9, 10**9, [1], [1])
print("SAI :", wrong_no_mod(*case), "| đúng:", max_area_sort(*case))

SAI : 999999998000000001 | đúng: 64


**Sửa:** `... % (10**9 + 7)`.

### Lỗi 5 — Ghép nhầm chiều

`horizontalCuts` đo từ **mép trên** ⇒ chia **chiều cao `h`**; `verticalCuts` chia **chiều rộng `w`**. Ghép `h` với `verticalCuts` là sai.

In [26]:
def wrong_swap(h, w, hc, vc):
    return max_gap(h, vc) * max_gap(w, hc) % MOD                    # SAI: ghép nhầm

h, w, hc, vc, want = examples[1]
print("SAI :", wrong_swap(h, w, hc, vc), "| đúng:", want)

SAI : 8 | đúng: 6


**Sửa:** `max_gap(h, horizontalCuts) * max_gap(w, verticalCuts)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Lưới tạo bởi các đường cắt **xuyên suốt** theo hai chiều ⇒ hai chiều độc lập ⇒ **tách thành hai bài 1D**.
- "Khoảng lớn nhất / nhỏ nhất giữa các điểm" ⇒ **sắp xếp + duyệt cặp kề**; nhớ thêm điểm biên.
- Thấy "trả về modulo `10^9+7`" trong bài tìm **max** ⇒ chỉ modulo khi trả về.
- Muốn `O(n)` cho khoảng lớn nhất ⇒ **chia xô / chuồng bồ câu** (bài 164).
- Tích hai số dương lớn nhất ⇒ lấy **max mỗi thừa số** (không đúng nếu có số âm!).

## 15. Biến thể của bài

1. **Miếng nhỏ nhất:** `min(H) · min(V)`.
2. **Đếm số miếng có diện tích lớn nhất:** (số dải ngang cao nhất) × (số dải dọc rộng nhất) — bài tập 3.
3. **164 — Maximum Gap:** khoảng lớn nhất giữa hai phần tử kề nhau sau sắp xếp, yêu cầu `O(n)`.
4. **Thêm nhát cắt trực tuyến (online):** sau mỗi nhát, báo khoảng lớn nhất hiện tại — xử lý ngược (xóa nhát cắt, gộp khoảng) bằng danh sách liên kết đôi, hoặc `SortedList` + đếm. (Bài tập 5.)
5. **Hình hộp 3D** cắt theo ba chiều: `maxH · maxW · maxD`.
6. **2943 — Maximize Area of Square Hole in Grid:** chọn bỏ bớt thanh để tạo lỗ vuông lớn nhất — cũng tách theo từng chiều.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê mọi miếng | Thử mọi cặp (dải ngang, dải dọc) | `O(A·B)` | `O(A+B)` | Kiểm chứng, input nhỏ |
| 2. Sort + max gap | `max(H)·max(V)` sau khi sắp | `O(A log A + B log B)` | `O(A+B)` | Mặc định — ngắn, nhanh trong Python |
| 3. Chia xô | Khoảng lớn nhất vắt qua ranh giới xô | `O(A+B)` | `O(A+B)` | Khi đề bắt buộc tuyến tính (bài 164) |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `maxArea(6, 6, [2, 4], [3])` bằng bao nhiêu?

**Bài 2 (tính tay):** Với `h = 7, w = 5, horizontalCuts = [6, 2], verticalCuts = [4, 1]`, liệt kê `H`, `V` và đáp án.

**Bài 3 (điền TODO):** Viết `count_max_pieces(h, w, hc, vc)` trả về **số miếng** có diện tích bằng diện tích lớn nhất.

**Bài 4 (tự viết lại):** Viết lại `max_gap_v2(L, cuts)` chỉ dùng **một biểu thức** `max(...)` với `zip` (không vòng `for` tường minh).

**Bài 5 (mini challenge):** Cho đoạn `[0, L]` và các nhát cắt được thêm **lần lượt** `cuts[0], cuts[1], …`. Viết `online_max_gap(L, cuts)` trả về danh sách: phần tử thứ `i` là khoảng lớn nhất sau khi thêm `i+1` nhát đầu tiên. Yêu cầu `O(n log n)` hoặc tốt hơn (gợi ý: xử lý ngược).

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_max_pieces(h: int, w: int, hc: List[int], vc: List[int]) -> int:
    # TODO: đếm số dải đạt max theo mỗi chiều rồi nhân
    raise NotImplementedError

# Bài 4 — TODO
def max_gap_v2(L: int, cuts: List[int]) -> int:
    # TODO: một dòng return max(... zip ...)
    raise NotImplementedError

# Bài 5 — TODO
def online_max_gap(L: int, cuts: List[int]) -> List[int]:
    # TODO: thêm hết nhát cắt, rồi xóa ngược từ cuối, gộp hai khoảng kề nhau
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ranh giới ngang `0,2,4,6`. (2) Ranh giới dọc `0,3,6`. (3) `2 × 3`.

**Bài 2.** (1) Sắp `[2, 6]`, `[1, 4]`. (2) `H = [2, 4, 1]`, `V = [1, 3, 1]`. (3) `4 × 3`.

**Bài 3.** (1) Dùng `strips` để có `H`, `V`. (2) `H.count(max(H))`. (3) Nhân hai số đếm.

**Bài 4.** (1) `p = [0, *sorted(cuts), L]`. (2) `zip(p, p[1:])`. (3) `return max(b - a for a, b in zip(p, p[1:]))`.

**Bài 5.** (1) Sau khi thêm **hết** nhát, tính các khoảng và khoảng lớn nhất. (2) Đi ngược: xóa nhát `cuts[i]` thì hai khoảng hai bên gộp lại; khoảng lớn nhất **chỉ có thể tăng** khi xóa ⇒ cập nhật `best = max(best, gộp)`. (3) Dùng `left[x]`, `right[x]` (danh sách liên kết đôi trên các điểm đã sắp) để biết hàng xóm trong `O(1)`; ghi đáp án trước khi xóa rồi đảo ngược danh sách.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, Bài 2
print("Bài 1:", max_area_sort(6, 6, [2, 4], [3]))
assert max_area_sort(6, 6, [2, 4], [3]) == 6
H2, V2 = strips(7, [6, 2]), strips(5, [4, 1])
print("Bài 2:", H2, V2, max_area_sort(7, 5, [6, 2], [4, 1]))
assert (H2, V2, max_area_sort(7, 5, [6, 2], [4, 1])) == ([2, 4, 1], [1, 3, 1], 12)

# Bài 3, Bài 4
def count_max_pieces_ans(h, w, hc, vc):
    H, V = strips(h, hc), strips(w, vc)
    return H.count(max(H)) * V.count(max(V))

def max_gap_v2_ans(L, cuts):
    p = [0, *sorted(cuts), L]
    return max(b - a for a, b in zip(p, p[1:]))

assert count_max_pieces_ans(5, 4, [3, 1], [1]) == 2          # ví dụ 2: hai miếng xanh và vàng
assert count_max_pieces_ans(6, 6, [2, 4], [3]) == 6
rng = random.Random(8)
for _ in range(500):
    h, w = rng.randint(2, 30), rng.randint(2, 30)
    hc = rng.sample(range(1, h), rng.randint(1, h - 1))
    vc = rng.sample(range(1, w), rng.randint(1, w - 1))
    areas = [x * y for x in strips(h, hc) for y in strips(w, vc)]
    assert count_max_pieces_ans(h, w, hc, vc) == areas.count(max(areas))
    assert max_gap_v2_ans(h, hc) == max_gap(h, hc)
print("Bài 3, 4: OK")

Bài 1: 6
Bài 2: [2, 4, 1] [1, 3, 1] 12
Bài 3, 4: OK


In [29]:
def online_max_gap_ans(L, cuts):
    pts = sorted([0, L, *cuts])
    idx = {x: i for i, x in enumerate(pts)}
    left = list(range(-1, len(pts) - 1))
    right = list(range(1, len(pts) + 1))
    best = max(b - a for a, b in zip(pts, pts[1:]))
    out = []
    for c in reversed(cuts):
        out.append(best)                    # đáp án khi nhát c còn đó
        i = idx[c]
        l, r = left[i], right[i]
        best = max(best, pts[r] - pts[l])   # gộp hai khoảng kề
        right[l], left[r] = r, l
    return out[::-1]

def online_brute(L, cuts):
    return [max_gap(L, cuts[:i + 1]) for i in range(len(cuts))]

assert online_max_gap_ans(10, [5, 2, 8]) == [5, 5, 3]
rng = random.Random(9)
for _ in range(500):
    L = rng.randint(2, 50)
    cuts = rng.sample(range(1, L), rng.randint(1, L - 1))
    assert online_max_gap_ans(L, cuts) == online_brute(L, cuts)
print("Bài 5:", online_max_gap_ans(10, [5, 2, 8]), "| OK 500 test")

Bài 5: [5, 5, 3] | OK 500 test


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
def gap(L, cuts):
    p = [0, *sorted(cuts), L]              # nhớ 2 biên + sắp xếp
    return max(b - a for a, b in zip(p, p[1:]))

return gap(h, horizontalCuts) * gap(w, verticalCuts) % (10**9 + 7)
```

**Checklist:**
- [ ] Đã **sắp xếp** cả hai mảng nhát cắt.
- [ ] Có tính dải sát **mép 0** và **mép h/w**.
- [ ] `horizontalCuts` đi với `h`, `verticalCuts` đi với `w`.
- [ ] Lấy max trên giá trị thật, **modulo ở cuối cùng**.
- [ ] Không quên `% (10**9 + 7)`.
- [ ] Đã thử: một nhát mỗi chiều, cắt sát mép (`1` hoặc `L−1`), `h = w = 10^9`.